# Configuración

In [ ]:
import torch

class Config:
    # --- RUTAS DE DATOS Y CHECKPOINTS ---
    
    # Directorio de los datos reales/balanceados (Usado para el orden de clases)
    TENSOR_DIR = "data/mel_tensors_20s"       
    
    # Directorio de los Pseudo-Labels (Usado en la Fase 2)
    PSEUDO_TENSOR_DIR = "data/mel_tensors_pseudo"  
    
    # Directorio del mejor modelo Estudiante (v2)
    CHECKPOINT_DIR = "checkpoints_v2"              
    
    # Directorio del Modelo Final Afinamiento (v3)
    FINETUNE_CHECKPOINT_DIR = "checkpoints_finetune"

    TEST_SPEC_DIR = "data/mel_tensors_test"
    
    # --- PARÁMETROS DE AUDIO Y MODELO ---
    MODEL_NAME = 'tf_efficientnet_b0_ns' 
    DURATION = 20           
    SR = 32000
    
    # --- HIPERPARÁMETROS DE ENTRENAMIENTO ---
    BATCH_SIZE = 16
    EPOCHS = 20
    WEIGHT_DECAY = 1e-4
    
    # FASE 2: Estudiante (LR estándar)
    LEARNING_RATE = 1e-3
    
    # FASE 3: Afinamiento (LR muy bajo para pulir pesos)
    FINETUNE_LR = 5e-06     
    FINETUNE_EPOCHS = 10    
    
    # MixUp (Estrategia Noisy Student)
    MIXUP_PROB = 0.6        
    MIXUP_ALPHA = 0.4       
    
    # --- HARDWARE Y VARIABLES DINÁMICAS ---
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    NUM_WORKERS = 4
    
    # Variables dinámicas (se definen al inicio de la ejecución)
    NUM_CLASSES = 0

# Model utils


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import numpy as np
import timm

# -------------------------------------------------------------------
# 1. DATASET ROBUSTO (Acepta listas de archivos para Split limpio)
# -------------------------------------------------------------------
class BirdDataset(Dataset):
    def __init__(self, file_list, class_to_idx, mixup_prob=0.0, alpha=0.4):
        """
        Args:
            file_list: Lista de tuplas (path_al_npy, label_str)
            class_to_idx: Diccionario fijo {'especie': 0, ...}
            mixup_prob: Float. 0.0 para validación, >0.0 para training.
        """
        self.file_list = file_list
        self.class_to_idx = class_to_idx
        self.mixup_prob = mixup_prob
        self.alpha = alpha
        self.num_classes = len(class_to_idx)

    def __len__(self):
        return len(self.file_list)

    def load_tensor(self, path):
        try:
            arr = np.load(path)
            return torch.tensor(arr, dtype=torch.float32)
        except Exception:
            # Fallback de seguridad: tensor de ceros
            return torch.zeros((128, 626)) 

    def get_one_hot(self, label_str):
        target = torch.zeros(self.num_classes)
        idx = self.class_to_idx[label_str]
        target[idx] = 1.0
        return target

    def __getitem__(self, idx):
        path, label_str = self.file_list[idx]
        
        # 1. Cargar dato
        X = self.load_tensor(path)
        y = self.get_one_hot(label_str)
        
        # 2. MixUp "On-the-Fly" (Solo se activa si mixup_prob > 0)
        if self.mixup_prob > 0 and np.random.random() < self.mixup_prob:
            # Elegir compañero aleatorio
            idx2 = np.random.randint(0, len(self.file_list))
            path2, label_str2 = self.file_list[idx2]
            
            X2 = self.load_tensor(path2)
            y2 = self.get_one_hot(label_str2)
            
            # Generar lambda de distribución Beta
            lam = np.random.beta(self.alpha, self.alpha)
            
            # Ajustar longitudes
            min_len = min(X.shape[1], X2.shape[1])
            X = lam * X[:, :min_len] + (1 - lam) * X2[:, :min_len]
            y = lam * y + (1 - lam) * y2
            
        # 3. Añadir dimensión de canal para EfficientNet (1, Freq, Time)
        X = X.unsqueeze(0) 
        return X, y

# -------------------------------------------------------------------
# [cite_start]2. BLOQUE DE ATENCIÓN (CAB-CNN del Paper 1) [cite: 8, 9, 51]
# -------------------------------------------------------------------
class ACB(nn.Module):
    def __init__(self, input_dim, num_classes, num_experts=8):
        super().__init__()
        # Expertos: N redes pequeñas en paralelo
        self.classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, 64),
                nn.ReLU(),
                nn.Linear(64, num_classes)
            ) for _ in range(num_experts)
        ])
        
        # Atención: Decide qué experto escuchar en cada instante t
        self.attention = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Tanh(),
            nn.Linear(128, num_experts),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        # x shape: (Batch, Time, Features)
        alpha = self.attention(x).unsqueeze(-1)          # (B, T, Experts, 1)
        
        # Ejecutar expertos
        outputs = [clf(x) for clf in self.classifiers]   # Lista de (B, T, Classes)
        outputs = torch.stack(outputs, dim=2)            # (B, T, Experts, Classes)
        
        # [cite_start]Suma ponderada: c_t = sum(alpha * predictions) [cite: 167]
        weighted_pred = torch.sum(outputs * alpha, dim=2) # (B, T, Classes)
        return weighted_pred

# -------------------------------------------------------------------
# 3. MODELO HÍBRIDO (EfficientNet Backbone + ACB Head)
# -------------------------------------------------------------------
class BirdCABModel(nn.Module):
    def __init__(self, num_classes, model_name):
        super().__init__()
        # [cite_start]Backbone del Ganador (Paper 2) [cite: 318, 321]
        self.backbone = timm.create_model(
            model_name, pretrained=True, num_classes=0, global_pool='', in_chans=1
        )
        
        # Inyección del bloque de atención
        self.acb = ACB(self.backbone.num_features, num_classes)

    def forward(self, x):
        # 1. Extracción de características (EfficientNet)
        x = self.backbone(x)           # (Batch, 1280, Freq, Time)
        
        # 2. Reducción de frecuencia (Promedio)
        x = torch.mean(x, dim=2)       # (Batch, 1280, Time)
        
        # 3. Preparar para ACB (Batch, Time, Features)
        x = x.permute(0, 2, 1)         
        
        # 4. Clasificación con Atención Framewise
        x = self.acb(x)                # (Batch, Time, Classes)
        
        # 5. Agregación temporal (Promedio final)
        return torch.mean(x, dim=1)    # (Batch, Classes)

# Train teacher 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score

# Importamos tus módulos locales
from config import Config
from model_utils import BirdDataset, BirdCABModel

def calculate_metrics(y_true, y_probs):
    """Función auxiliar para calcular métricas de forma segura"""
    # Binarizar predicciones para F1 (Threshold 0.5)
    y_pred_binary = (y_probs > 0.5).astype(int)
    
    # En caso de MixUp, y_true puede tener decimales. Lo binarizamos para F1
    y_true_binary = (y_true > 0.5).astype(int)

    try:
        # AUC (Macro)
        auc = roc_auc_score(y_true_binary, y_probs, average='macro')
        # cmAP (Class-mean Average Precision)
        cmap = average_precision_score(y_true_binary, y_probs, average='macro')
        # F1-Score (Macro)
        f1 = f1_score(y_true_binary, y_pred_binary, average='macro')
    except ValueError:
        # Manejo de error si falta alguna clase en el batch
        auc, cmap, f1 = 0.5, 0.0, 0.0
        
    return auc, cmap, f1

def train():
    # ==============================================================================
    # 1. PREPARACIÓN DE DATOS
    # ==============================================================================
    print(f"🔍 Escaneando dataset en: {Config.TENSOR_DIR}...")
    root = Path(Config.TENSOR_DIR)
    
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    if not classes: raise ValueError("❌ No se encontraron clases.")

    class_to_idx = {cls: i for i, cls in enumerate(classes)}
    Config.NUM_CLASSES = len(classes)
    print(f"✔ Detectadas {Config.NUM_CLASSES} especies.")

    master_list = []
    stems = []  
    for cls in classes:
        for f in (root / cls).glob("*.npy"):
            stem = f.stem.split("_seg")[0] 
            master_list.append((str(f), cls))
            stems.append(stem)
            
    # SPLIT POR GRUPOS
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(gss.split(master_list, groups=stems))
    
    train_files = [master_list[i] for i in train_idx]
    val_files = [master_list[i] for i in val_idx]
    
    train_ds = BirdDataset(train_files, class_to_idx, mixup_prob=Config.MIXUP_PROB, alpha=Config.MIXUP_ALPHA)
    val_ds = BirdDataset(val_files, class_to_idx, mixup_prob=0.0)
    
    loaders = {
        'train': DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS),
        'val': DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
    }

    # ==============================================================================
    # 2. BALANCEO Y MODELO
    # ==============================================================================
    print("⚖️ Calculando pesos...")
    label_counts = np.zeros(Config.NUM_CLASSES)
    for _, label_str in train_files:
        label_counts[class_to_idx[label_str]] += 1
    
    pos_weights = np.max(label_counts) / np.maximum(label_counts, 1)
    pos_weights = torch.tensor(pos_weights, dtype=torch.float32).to(Config.DEVICE)
    
    print(f"🏗 Inicializando modelo...")
    model = BirdCABModel(Config.NUM_CLASSES, Config.MODEL_NAME).to(Config.DEVICE)
    
    optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=Config.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

    # ==============================================================================
    # 3. ENTRENAMIENTO
    # ==============================================================================
    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
    best_cmap = 0.0 
    
    print("🚀 ¡Iniciando entrenamiento!")
    
    for epoch in range(Config.EPOCHS):
        
        # --- TRAIN LOOP ---
        model.train()
        train_loss = 0.0
        
        # Listas para métricas de TRAIN
        train_probs = []
        train_targets = []

        loop = tqdm(loaders['train'], desc=f"Ep {epoch+1} [Train]")
        
        for X, y in loop:
            X, y = X.to(Config.DEVICE), y.to(Config.DEVICE)
            
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            # Guardar predicciones para métricas (detach para no guardar grafos en memoria)
            train_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            train_targets.append(y.detach().cpu().numpy())
            
            loop.set_postfix(loss=f"{loss.item():.4f}")

        # --- VAL LOOP ---
        model.eval()
        val_loss = 0.0
        val_probs = []
        val_targets = []
        
        with torch.no_grad():
            for X, y in loaders['val']:
                X, y = X.to(Config.DEVICE), y.to(Config.DEVICE)
                logits = model(X)
                loss = criterion(logits, y)
                val_loss += loss.item()
                
                val_probs.append(torch.sigmoid(logits).cpu().numpy())
                val_targets.append(y.cpu().numpy())

        # --- CÁLCULO DE MÉTRICAS GLOBALES ---
        
        # 1. Concatenar todo (Train)
        train_probs = np.vstack(train_probs)
        train_targets = np.vstack(train_targets)
        t_auc, t_cmap, t_f1 = calculate_metrics(train_targets, train_probs)
        
        # 2. Concatenar todo (Val)
        val_probs = np.vstack(val_probs)
        val_targets = np.vstack(val_targets)
        v_auc, v_cmap, v_f1 = calculate_metrics(val_targets, val_probs)

        # Promedios de Loss
        avg_train_loss = train_loss / len(loaders['train'])
        avg_val_loss = val_loss / len(loaders['val'])
        
        # --- REPORTE ---
        print(f"\n🏁 Resumen Ep {epoch+1} | LR: {optimizer.param_groups[0]['lr']:.1e}")
        print(f"   📘 TRAIN -> Loss: {avg_train_loss:.4f} | AUC: {t_auc:.4f} | cmAP: {t_cmap:.4f} | F1: {t_f1:.4f}")
        print(f"   📗 VAL   -> Loss: {avg_val_loss:.4f} | AUC: {v_auc:.4f} | cmAP: {v_cmap:.4f} | F1: {v_f1:.4f}")
        
        if v_cmap > best_cmap:
            best_cmap = v_cmap
            torch.save(model.state_dict(), f"{Config.CHECKPOINT_DIR}/best_model_cmap.pth")
            print(f"   🏆 ¡Nuevo Récord! (cmAP: {v_cmap:.4f})")
            
        scheduler.step(v_cmap)
        print("-" * 60)

if __name__ == "__main__":
    train()

# inference pseudo

In [ ]:
import torch
import numpy as np
import librosa
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd

# Importamos tu configuración y modelo
from config import Config
from model_utils import BirdCABModel

# --- CONFIGURACIÓN DE INFERENCIA ---
# Carpeta donde tengas audios largos sin etiqueta (wav, ogg, mp3)
SOUNDSCAPES_DIR = "data/train_soundscapes" 
# Donde guardaremos los nuevos "tesoros" encontrados
PSEUDO_LABEL_DIR = "data/mel_tensors_pseudo"
# Umbral estricto: Solo confiamos si la probabilidad es > 95%
CONFIDENCE_THRESHOLD = 0.95 

# Parámetros de audio (Deben ser IDÉNTICOS al entrenamiento)
SR = 32000
DURATION = 20
HOP_LENGTH = 5  # Avance de 5s (Solapamiento para no perder nada)

def compute_melspec(y, sr):
    """Generación de espectrograma idéntica a la etapa 1"""
    melspec = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=128, fmin=20, fmax=16000
    )
    return librosa.power_to_db(melspec).astype(np.float32)

def generate_pseudo_labels():
    # 1. Cargar el modelo entrenado (El mejor guardado)
    print(f" Cargando el mejor modelo desde {Config.CHECKPOINT_DIR}...")
    
    # Necesitamos saber cuántas clases había para inicializar la arquitectura
    # Leemos las carpetas de entrenamiento para recuperar el orden de clases
    train_root = Path(Config.TENSOR_DIR)
    classes = sorted([d.name for d in train_root.iterdir() if d.is_dir()])
    Config.NUM_CLASSES = len(classes)
    idx_to_class = {i: cls for i, cls in enumerate(classes)}
    
    model = BirdCABModel(Config.NUM_CLASSES, Config.MODEL_NAME).to(Config.DEVICE)
    
    # Cargar pesos
    checkpoint_path = f"{Config.CHECKPOINT_DIR}/best_model_cmap.pth"
    model.load_state_dict(torch.load(checkpoint_path, map_location=Config.DEVICE))
    model.eval()
    
    print(f"✔ Modelo cargado. Clases: {Config.NUM_CLASSES}. Umbral: {CONFIDENCE_THRESHOLD}")
    print(f"🔍 Escaneando soundscapes en {SOUNDSCAPES_DIR}...")
    
    os.makedirs(PSEUDO_LABEL_DIR, exist_ok=True)
    audio_files = list(Path(SOUNDSCAPES_DIR).glob("*.ogg")) + list(Path(SOUNDSCAPES_DIR).glob("*.wav"))
    
    total_found = 0
    
    # 2. Procesar cada archivo de audio largo
    for audio_path in tqdm(audio_files, desc="Analizando audios"):
        try:
            # Cargar audio completo
            y, _ = librosa.load(audio_path, sr=SR)
            
            # Cortar en ventanas deslizantes
            # Ventana de 20s, avanzando cada 5s
            step = int(HOP_LENGTH * SR)
            window = int(DURATION * SR)
            
            for start_sample in range(0, len(y) - window, step):
                chunk = y[start_sample : start_sample + window]
                
                # Generar Tensor
                mel = compute_melspec(chunk, SR)
                tensor = torch.tensor(mel).unsqueeze(0).unsqueeze(0) # (1, 1, Freq, Time)
                
                # Inferencia
                with torch.no_grad():
                    tensor = tensor.to(Config.DEVICE)
                    logits = model(tensor)
                    probs = torch.sigmoid(logits) # Convertir a probabilidad (0-1)
                
                # 3. FILTRADO (La parte clave para evitar ruido)
                max_prob, pred_idx = torch.max(probs, dim=1)
                confidence = max_prob.item()
                
                if confidence > CONFIDENCE_THRESHOLD:
                    # ¡Bingo! El modelo está muy seguro de que esto es un pájaro conocido
                    pred_class = idx_to_class[pred_idx.item()]
                    
                    # Guardar como nuevo dato de entrenamiento
                    save_dir = Path(PSEUDO_LABEL_DIR) / pred_class
                    save_dir.mkdir(parents=True, exist_ok=True)
                    
                    filename = f"pseudo_{audio_path.stem}_{start_sample}.npy"
                    np.save(save_dir / filename, mel)
                    
                    total_found += 1
                    
        except Exception as e:
            print(f"Error procesando {audio_path.name}: {e}")

    print("\n" + "="*50)
    print(f" Proceso terminado.")
    print(f" Se encontraron {total_found} nuevos segmentos de alta confianza.")
    print(f" Guardados en: {PSEUDO_LABEL_DIR}")
    print("="*50)

if __name__ == "__main__":
    generate_pseudo_labels()

# Train student


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score

# Importamos tus módulos
from config import Config
from model_utils import BirdDataset, BirdCABModel

def calculate_metrics(y_true, y_probs):
    """Calcula AUC, cmAP y F1"""
    # Binarizar para F1
    y_pred_binary = (y_probs > 0.5).astype(int)
    y_true_binary = (y_true > 0.5).astype(int)
    
    try:
        # AUC Macro
        auc = roc_auc_score(y_true_binary, y_probs, average='macro')
        # cmAP (Métrica principal de BirdCLEF)
        cmap = average_precision_score(y_true_binary, y_probs, average='macro')
        # F1 Score Macro
        f1 = f1_score(y_true_binary, y_pred_binary, average='macro')
    except ValueError:
        # Si falta una clase en el batch, retornamos 0 para no romper el loop
        auc, cmap, f1 = 0.5, 0.0, 0.0
    return auc, cmap, f1

def scan_folder(folder_path):
    """Escanea una carpeta y devuelve lista de archivos + stems"""
    file_list = []
    stems = []
    root = Path(folder_path)
    
    # --- CORRECCIÓN DEL ERROR ---
    if not root.exists():
        print(f"⚠️  ADVERTENCIA CRÍTICA: No se encontró la carpeta: {folder_path}")
        print(f"    ¿Estás ejecutando el script desde la carpeta correcta?")
        # Devolvemos 3 valores vacíos para evitar el ValueError
        return [], [], []
        
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    
    for cls in classes:
        for f in (root / cls).glob("*.npy"):
            # Lógica para extraer el ID de la grabación y evitar fugas
            stem = f.stem.split("_seg")[0] if "_seg" in f.name else f.stem.rsplit("_", 1)[0]
            file_list.append((str(f), cls))
            stems.append(stem)
            
    return file_list, stems, classes

def train():
    # ==============================================================================
    # 1. CARGA DE DATOS
    # ==============================================================================
    print("🚀 FASE 2: Entrenando al Estudiante (Noisy Student)...")
    
    # A) Escanear Datos REALES
    print(f"🔍 Escaneando REALES: {Config.TENSOR_DIR}...")
    real_files, real_stems, real_classes = scan_folder(Config.TENSOR_DIR)
    
    if not real_files: 
        print("❌ ERROR: No hay archivos reales. Revisa la ruta en config.py")
        return # Detenemos si no hay datos
    
    Config.NUM_CLASSES = len(real_classes)
    class_to_idx = {cls: i for i, cls in enumerate(real_classes)}
    print(f"✔ Clases Reales: {Config.NUM_CLASSES}")

    # B) Escanear Datos PSEUDO
    print(f"🔍 Escaneando PSEUDO: {Config.PSEUDO_TENSOR_DIR}...")
    pseudo_files, _, _ = scan_folder(Config.PSEUDO_TENSOR_DIR)
    print(f"✔ Datos Pseudo encontrados: {len(pseudo_files)}")
    
    # C) SPLIT (Solo sobre reales)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(gss.split(real_files, groups=real_stems))
    
    real_train = [real_files[i] for i in train_idx]
    real_val = [real_files[i] for i in val_idx]
    
    # D) FUSIÓN: Train = Reales + Pseudo
    final_train_list = real_train + pseudo_files
    
    print(f"📊 Dataset Final -> Train: {len(final_train_list)} | Val: {len(real_val)}")

    # Crear Datasets
    train_ds = BirdDataset(final_train_list, class_to_idx, mixup_prob=Config.MIXUP_PROB, alpha=Config.MIXUP_ALPHA)
    val_ds = BirdDataset(real_val, class_to_idx, mixup_prob=0.0) 
    
    loaders = {
        'train': DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS),
        'val': DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
    }

    # ==============================================================================
    # 2. CONFIGURACIÓN
    # ==============================================================================
    print("⚖️ Calculando pesos (basado en datos reales)...")
    label_counts = np.zeros(Config.NUM_CLASSES)
    for _, label_str in real_train:
        if label_str in class_to_idx:
            label_counts[class_to_idx[label_str]] += 1
            
    pos_weights = np.max(label_counts) / np.maximum(label_counts, 1)
    pos_weights = torch.tensor(pos_weights, dtype=torch.float32).to(Config.DEVICE)
    
    print(f"🏗 Inicializando modelo Estudiante...")
    model = BirdCABModel(Config.NUM_CLASSES, Config.MODEL_NAME).to(Config.DEVICE)
    
    optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=Config.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

    # ==============================================================================
    # 3. ENTRENAMIENTO
    # ==============================================================================
    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
    best_cmap = 0.0 
    
    print("🚀 ¡Despegue V2!")
    
    for epoch in range(Config.EPOCHS):
        # --- TRAIN ---
        model.train()
        train_loss = 0.0
        train_probs, train_targets = [], []

        loop = tqdm(loaders['train'], desc=f"Ep {epoch+1} [Student]")
        
        for X, y in loop:
            X, y = X.to(Config.DEVICE), y.to(Config.DEVICE)
            
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            # Guardamos predicciones para calcular métricas de train
            train_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            train_targets.append(y.detach().cpu().numpy())
            loop.set_postfix(loss=f"{loss.item():.4f}")

        # --- VALIDATION ---
        model.eval()
        val_loss = 0.0
        val_probs, val_targets = [], []
        
        with torch.no_grad():
            for X, y in loaders['val']:
                X, y = X.to(Config.DEVICE), y.to(Config.DEVICE)
                logits = model(X)
                loss = criterion(logits, y)
                val_loss += loss.item()
                val_probs.append(torch.sigmoid(logits).cpu().numpy())
                val_targets.append(y.cpu().numpy())

        # --- CÁLCULO DE MÉTRICAS ---
        # Train
        train_probs = np.vstack(train_probs)
        train_targets = np.vstack(train_targets)
        t_auc, t_cmap, t_f1 = calculate_metrics(train_targets, train_probs)
        
        # Val
        val_probs = np.vstack(val_probs)
        val_targets = np.vstack(val_targets)
        v_auc, v_cmap, v_f1 = calculate_metrics(val_targets, val_probs)

        avg_train_loss = train_loss / len(loaders['train'])
        avg_val_loss = val_loss / len(loaders['val'])
        
        # --- IMPRESIÓN DETALLADA ---
        print(f"\n🏁 Resumen Ep {epoch+1} | LR: {optimizer.param_groups[0]['lr']:.1e}")
        print(f"   📘 TRAIN -> Loss: {avg_train_loss:.4f} | cmAP: {t_cmap:.4f} | AUC: {t_auc:.4f} | F1: {t_f1:.4f}")
        print(f"   📗 VAL   -> Loss: {avg_val_loss:.4f} | cmAP: {v_cmap:.4f} | AUC: {v_auc:.4f} | F1: {v_f1:.4f}")
        
        if v_cmap > best_cmap:
            best_cmap = v_cmap
            torch.save(model.state_dict(), f"{Config.CHECKPOINT_DIR}/best_student.pth")
            print(f"   🏆 ¡Nuevo Récord del Estudiante! ({v_cmap:.4f})")
            
        scheduler.step(v_cmap)
        print("-" * 60)

if __name__ == "__main__":
    train()

# fine tuning


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import os
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score

# Importamos tus módulos
from config import Config
from model_utils import BirdCABModel, BirdDataset
from train2 import calculate_metrics, scan_folder # Reutilizamos las funciones

def train_finetune():
    # ==============================================================================
    # 1. PREPARACIÓN DE DATOS (Solo Reales)
    # ==============================================================================
    print("🔬 FASE 3: Afinamiento de Precisión...")
    print(f"🔍 Escaneando REALES: {Config.TENSOR_DIR}...")
    
    # 1. Cargamos el set de datos real (balanced)
    real_files, real_stems, real_classes = scan_folder(Config.TENSOR_DIR)
    
    if not real_files: 
        print("❌ ERROR: No hay datos reales para el afinamiento.")
        return 
        
    Config.NUM_CLASSES = len(real_classes)
    class_to_idx = {cls: i for i, cls in enumerate(real_classes)}

    # 2. Split por Grupos para Train/Val
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(gss.split(real_files, groups=real_stems))
    
    train_clean_files = [real_files[i] for i in train_idx]
    val_clean_files = [real_files[i] for i in val_idx]
    
    print(f"📊 Split Final -> Train: {len(train_clean_files)} | Val: {len(val_clean_files)}")

    # Crear Datasets (Solo datos REALES y MixUp desactivado o muy bajo)
    train_ds = BirdDataset(train_clean_files, class_to_idx, mixup_prob=0.0) # MixUp Desactivado
    val_ds = BirdDataset(val_clean_files, class_to_idx, mixup_prob=0.0) 
    
    loaders = {
        'train': DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS),
        'val': DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
    }

    # ==============================================================================
    # 2. CARGA DE PESOS Y CONFIGURACIÓN (LR Baja)
    # ==============================================================================
    
    # Calculamos pesos solo en el set de entrenamiento limpio
    label_counts = np.zeros(Config.NUM_CLASSES)
    for _, label_str in train_clean_files:
        if label_str in class_to_idx:
            label_counts[class_to_idx[label_str]] += 1
            
    pos_weights = np.max(label_counts) / np.maximum(label_counts, 1)
    pos_weights = torch.tensor(pos_weights, dtype=torch.float32).to(Config.DEVICE)

    print("🏗 Inicializando modelo y cargando pesos de 'best_student.pth'...")
    model = BirdCABModel(Config.NUM_CLASSES, Config.MODEL_NAME).to(Config.DEVICE)
    
    # --- CARGA CRUCIAL DE PESOS ---
    student_checkpoint_path = f"{Config.CHECKPOINT_DIR}/best_student.pth"
    if os.path.exists(student_checkpoint_path):
        model.load_state_dict(torch.load(student_checkpoint_path, map_location=Config.DEVICE))
    else:
        print("❌ ERROR: No se encontró best_student.pth. Iniciando desde cero.")
        return

    # Usamos el LR de afinamiento
    optimizer = optim.AdamW(model.parameters(), lr=Config.FINETUNE_LR, weight_decay=Config.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

    # ==============================================================================
    # 3. BUCLE DE AFINAMIENTO
    # ==============================================================================
    finetune_checkpoint_dir = "checkpoints_finetune"
    os.makedirs(finetune_checkpoint_dir, exist_ok=True)
    best_cmap = 0.0 
    
    print(f"⭐ Iniciando Afinamiento por {Config.FINETUNE_EPOCHS} épocas...")
    
    for epoch in range(Config.FINETUNE_EPOCHS):
        # --- TRAIN (Con LR bajo) ---
        model.train()
        train_loss = 0.0
        train_probs, train_targets = [], []
        
        loop = tqdm(loaders['train'], desc=f"Ep {epoch+1}/{Config.FINETUNE_EPOCHS} [Afinamiento]")
        
        for X, y in loop:
            X, y = X.to(Config.DEVICE), y.to(Config.DEVICE)
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            train_targets.append(y.detach().cpu().numpy())
            loop.set_postfix(loss=f"{loss.item():.4f}")

        # --- VALIDATION (Pura) ---
        model.eval()
        val_loss = 0.0
        val_probs, val_targets = [], []
        
        with torch.no_grad():
            for X, y in loaders['val']:
                X, y = X.to(Config.DEVICE), y.to(Config.DEVICE)
                val_loss += criterion(model(X), y).item()
                val_probs.append(torch.sigmoid(model(X)).cpu().numpy())
                val_targets.append(y.cpu().numpy())

        # Metrics
        t_auc, t_cmap, t_f1 = calculate_metrics(np.vstack(train_targets), np.vstack(train_probs))
        v_auc, v_cmap, v_f1 = calculate_metrics(np.vstack(val_targets), np.vstack(val_probs))

        avg_train = train_loss / len(loaders['train'])
        avg_val = val_loss / len(loaders['val'])
        
        # Reporte
        print(f"\n🏁 Resumen Ep {epoch+1} | LR: {optimizer.param_groups[0]['lr']:.1e}")
        print(f"   📘 TRAIN -> Loss: {avg_train:.4f} | cmAP: {t_cmap:.4f} | F1: {t_f1:.4f}")
        print(f"   📗 VAL   -> Loss: {avg_val:.4f} | cmAP: {v_cmap:.4f} | F1: {v_f1:.4f}")
        
        if v_cmap > best_cmap:
            best_cmap = v_cmap
            torch.save(model.state_dict(), f"{finetune_checkpoint_dir}/best_finetune.pth")
            print(f"   🏆 ¡Récord Final! cmAP: {v_cmap:.4f}")
            
        scheduler.step(v_cmap)
        print("-" * 60)

if __name__ == "__main__":
    # Nota: Asegúrate de que train2.py y config.py están en el path de importación.
    try:
        from train2 import calculate_metrics, scan_folder 
    except ImportError:
        # En caso de que el import fallara, definimos las funciones aquí manualmente.
        # Esto es solo un fallback de seguridad.
        print("⚠️ Advertencia: No se pudo importar de train2.py. Asegúrate de tener las funciones calculate_metrics y scan_folder definidas correctamente.")
        # Aquí se debería poner el código de las funciones manualmente si el import falla.
        # Dado que el usuario tiene los archivos, confiaremos en la importación.
        pass
        
    train_finetune()